# Deploying Iris-detection model using Vertex AI


### Dataset

This tutorial uses R.A. Fisher's Iris dataset, a small and popular dataset for machine learning experiments. Each instance has four numerical features, which are different measurements of a flower, and a target label that
categorizes the flower into: **Iris setosa**, **Iris versicolour** and **Iris virginica**.

This tutorial uses [a version of the Iris dataset available in the
scikit-learn library](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_iris.html#sklearn.datasets.load_iris).

## Get started

### Install Vertex AI SDK for Python and other required packages



In [13]:

# Vertex SDK for Python
! pip3 install --upgrade --quiet  google-cloud-aiplatform

### Set Google Cloud project information
Learn more about [setting up a project and a development environment](https://cloud.google.com/vertex-ai/docs/start/cloud-environment).

In [14]:
PROJECT_ID = "nifty-harmony-474217-q6"  # @param {type:"string"}
LOCATION = "us-central1"  # @param {type:"string"}

### Create a Cloud Storage bucket

Create a storage bucket to store intermediate artifacts such as datasets.

In [13]:
BUCKET_URI = f"gs://mlops-course-nifty-harmony-474217-q6-unique"  # @param {type:"string"}

**If your bucket doesn't already exist**: Run the following cell to create your Cloud Storage bucket.

In [4]:
! gsutil mb -l {LOCATION} -p {PROJECT_ID} {BUCKET_URI}

Creating gs://mlops-course-nifty-harmony-474217-q6-unique/...
ServiceException: 409 A Cloud Storage bucket named 'mlops-course-nifty-harmony-474217-q6-unique' already exists. Try another name. Bucket names must be globally unique across all Google Cloud projects, including those outside of your organization.


### Initialize Vertex AI SDK for Python

To get started using Vertex AI, you must have an existing Google Cloud project and [enable the Vertex AI API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com).

In [16]:
from google.cloud import aiplatform

aiplatform.init(project=PROJECT_ID, location=LOCATION, staging_bucket=BUCKET_URI)

### Import the required libraries

In [17]:
import os
import sys

### Configure resource names

Set a name for the following parameters:

`MODEL_ARTIFACT_DIR` - Folder directory path to your model artifacts within a Cloud Storage bucket, for example: "my-models/fraud-detection/trial-4"

`REPOSITORY` - Name of the Artifact Repository to create or use.

`IMAGE` - Name of the container image that is pushed to the repository.

`MODEL_DISPLAY_NAME` - Display name of Vertex AI model resource.

In [18]:
MODEL_ARTIFACT_DIR = "my-models/iris-classifier-week-1"  # @param {type:"string"}
REPOSITORY = "iris-classifier-repo"  # @param {type:"string"}
IMAGE = "iris-classifier-img"  # @param {type:"string"}
MODEL_DISPLAY_NAME = "iris-classifier"  # @param {type:"string"}

# Set the defaults if no names were specified
if MODEL_ARTIFACT_DIR == "[your-artifact-directory]":
    MODEL_ARTIFACT_DIR = "custom-container-prediction-model"

if REPOSITORY == "[your-repository-name]":
    REPOSITORY = "custom-container-prediction"

if IMAGE == "[your-image-name]":
    IMAGE = "sklearn-fastapi-server"

if MODEL_DISPLAY_NAME == "[your-model-display-name]":
    MODEL_DISPLAY_NAME = "sklearn-custom-container"

# Homework Pipeline

## Requirement 1: Importing data from the Google Storage Bucket

In [18]:
from google.cloud import storage
import pandas as pd

TRAINING_DATA_BUCKET_NAME="training_data_mlops_w1"
TRAINING_BLOB="iris.csv"

def load_iris_from_gc(bucket_name=TRAINING_DATA_BUCKET_NAME,blob_name=TRAINING_BLOB):
    client=storage.Client()
    bucket=client.bucket(bucket_name)
    blob=bucket.blob(blob_name)
    data_bytes=blob.download_as_bytes()
    df=pd.read_csv(pd.io.common.BytesIO(data_bytes))
    return df

iris_df=load_iris_from_gc()
data=iris_df
iris_df.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


## Simple Decision Tree model
Build a Decision Tree model on iris data

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from pandas.plotting import parallel_coordinates
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn import metrics

In [8]:
train, test = train_test_split(data, test_size = 0.4, stratify = data['species'], random_state = 42)
X_train = train[['sepal_length','sepal_width','petal_length','petal_width']]
y_train = train.species
X_test = test[['sepal_length','sepal_width','petal_length','petal_width']]
y_test = test.species

In [9]:
mod_dt = DecisionTreeClassifier(max_depth = 3, random_state = 1)
mod_dt.fit(X_train,y_train)
prediction=mod_dt.predict(X_test)
print('The accuracy of the Decision Tree is',"{:.3f}".format(metrics.accuracy_score(prediction,y_test)))

The accuracy of the Decision Tree is 0.983


In [10]:
import pickle
import joblib

joblib.dump(mod_dt, "artifacts/model.joblib")

['artifacts/model.joblib']

### Storing output artifact in the artifact bucket

In [30]:
import datetime

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
!gsutil cp artifacts/model.joblib gs://mlops-course-nifty-harmony-474217-q6-unique/my-models/iris-classifier-week-1/{timestamp}/model.joblib
print(f"gs://training_data_mlops_w1/artifacts/{timestamp}/model.joblib")


Copying file://artifacts/model.joblib [Content-Type=application/octet-stream]...
/ [1 files][  2.5 KiB/  2.5 KiB]                                                
Operation completed over 1 objects/2.5 KiB.                                      
gs://training_data_mlops_w1/artifacts/20251012_055414/model.joblib


### Inference Script

#### imports

In [23]:
from google.cloud import storage
import joblib
import os

In [20]:
{BUCKET_URI}

{'gs://mlops-course-nifty-harmony-474217-q6-unique'}

In [38]:
from google.cloud import storage
import joblib

bucket_name = "mlops-course-nifty-harmony-474217-q6-unique"
model_dir = "my-models/iris-classifier-week-1/"

client = storage.Client()
bucket = client.bucket(bucket_name)
blobs = list(bucket.list_blobs(prefix=model_dir))
model_files = [blob.name for blob in blobs if blob.name.endswith("model.joblib")]
model_files.sort()
latest_model_path = model_files[-1]

blob = bucket.blob(latest_model_path)
blob.download_to_filename("latest_model.joblib")

model = joblib.load("latest_model.joblib")
print("Latest model loaded")


Latest model loaded


In [39]:
prediction=mod_dt.predict(X_test)
print('The accuracy of the Decision Tree is',"{:.3f}".format(metrics.accuracy_score(prediction,y_test)))

The accuracy of the Decision Tree is 0.983


# Week 2

In [2]:
!pip install dvc

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 46.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 98.0 MB/s  0:00:00
  DEPRECATION: Building 'antlr4-python3-runtime' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'antlr4-python3-runtime'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  Created wheel for antlr4-python3-runtime: filename=antlr4_python3_runtime-4.9.3-py3-none-any.whl size=144591 sha256=bdf42bd5385895a3ca334c78a812dd101e03a23aa3f73f0caf844ee718e77bd3
  Stored in directory: /home/jupyter/.cache/pip/wheels/12/93/dd/1f6a127edc45659556564c5730f6d4e300888f4bca2d4c5a88
Successfully built antlr4

In [3]:
!git init
!dvc init
!dvc remote add -d myremote gs://training_data_mlops_w1

Reinitialized existing Git repository in /home/jupyter/.git/
Initialized DVC repository.

You can now commit the changes to git.

+---------------------------------------------------------------------+
|                                                                     |
|        DVC has enabled anonymous aggregate usage analytics.         |
|     Read the analytics documentation (and how to opt-out) here:     |
|             <https://dvc.org/doc/user-guide/analytics>              |
|                                                                     |
+---------------------------------------------------------------------+

What's next?
------------
- Check out the documentation: <https://dvc.org/doc>
- Get help and share ideas: <https://dvc.org/chat>
- Star us on GitHub: <https://github.com/iterative/dvc>
Setting 'myremote' as a default remote.


In [6]:
!dvc remote add -d -f myremote gs://training_data_mlops_w1

Setting 'myremote' as a default remote.


In [12]:
!git add Homework_Pipeline.ipynb

In [13]:
!dvc add iris.csv

 ⠋ Checking graph
Adding...                                                                       
!
                                                                                
!
  0% Checking cache in '/home/jupyter/.dvc/cache/files/md5'| |0/? [00:00<?,    ?
                                                                                
!
  0%|          |Adding iris.csv to cache              0/1 [00:00<?,     ?file/s]
                                                                                
!
  0%|          |Checking out /home/jupyter/iris.csv   0/1 [00:00<?,    ?files/s]
100% Adding...|████████████████████████████████████████|1/1 [00:00, 18.70file/s]

To track the changes with git, run:

	git add .gitignore iris.csv.dvc

To enable auto staging, run:

	dvc config core.autostage true


In [14]:
!git add .gitignore iris.csv.dvc

In [16]:
!git commit -m "Clean tracking: Git for code, DVC for data"

[master 58a658c] Clean tracking: Git for code, DVC for data
 85 files changed, 964 insertions(+), 5385 deletions(-)
 delete mode 100644 .bashrc
 delete mode 100644 .cache/matplotlib/fontlist-v390.json
 delete mode 100644 .cache/pip/http-v2/6/3/b/4/6/63b46067f6accfebc4ed39f857e516856532d58972257ecafa23dc5c
 delete mode 100644 .cache/pip/http-v2/6/3/b/4/6/63b46067f6accfebc4ed39f857e516856532d58972257ecafa23dc5c.body
 delete mode 100644 .cache/pip/http-v2/a/1/9/5/3/a19537d3cf37c122db841d6fe4cd322bc10d1a558bb00d146b85cb9a
 delete mode 100644 .cache/pip/http-v2/a/1/9/5/3/a19537d3cf37c122db841d6fe4cd322bc10d1a558bb00d146b85cb9a.body
 delete mode 100644 .cache/pip/selfcheck/ab2bb442149194bcbab5700d8f932112632ba9320db09910f4a4422b
 delete mode 100644 .config/gcloud/.last_survey_prompt.yaml
 delete mode 100644 .config/gcloud/access_tokens.db
 delete mode 100644 .config/gcloud/active_config
 delete mode 100644 .config/gcloud/config_sentinel
 delete mode 100644 .config/gcloud/configurations/confi

## Augmenting IRIS Data

In [19]:
# Get existing data
import pandas as pd
from sklearn.datasets import load_iris

iris_df_v1=load_iris_from_gc()
iris_df_v1.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [29]:
len(iris_df_v1)

150

In [34]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from pandas.plotting import parallel_coordinates
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn import metrics

In [35]:
train, test = train_test_split(data, test_size = 0.4, stratify = data['species'], random_state = 42)
X_train = train[['sepal_length','sepal_width','petal_length','petal_width']]
y_train = train.species
X_test = test[['sepal_length','sepal_width','petal_length','petal_width']]
y_test = test.species

In [36]:
mod_dt = DecisionTreeClassifier(max_depth = 3, random_state = 1)
mod_dt.fit(X_train,y_train)
prediction=mod_dt.predict(X_test)
print('The accuracy of the Decision Tree is',"{:.3f}".format(metrics.accuracy_score(prediction,y_test)))

The accuracy of the Decision Tree is 0.983


In [37]:
iris_df_v1.to_csv("iris.csv",index=False)
!dvc add iris.csv

 ⠋ Checking graph
Adding...                                                                       
!
                                                                                
!
  0% Checking cache in '/home/jupyter/.dvc/cache/files/md5'| |0/? [00:00<?,    ?
                                                                                
!
  0%|          |Checking out /home/jupyter/iris.csv   0/1 [00:00<?,    ?files/s]
100% Adding...|████████████████████████████████████████|1/1 [00:00, 25.44file/s]

To track the changes with git, run:

	git add iris.csv.dvc

To enable auto staging, run:

	dvc config core.autostage true


In [39]:
import pickle
import joblib

joblib.dump(mod_dt, "artifacts/model.joblib")

['artifacts/model.joblib']

In [43]:
!dvc add artifacts/model.joblib
!git add artifacts/.gitignore artifacts/model.joblib.dvc
!git commit -m "Trained model 1"

 ⠋ Checking graph
Adding...                                                                       
!
                                                                                
!
  0% Checking cache in '/home/jupyter/.dvc/cache/files/md5'| |0/? [00:00<?,    ?
                                                                                
!
  0%|          |Checking out /home/jupyter/artifacts/m0/1 [00:00<?,    ?files/s]
100% Adding...|████████████████████████████████████████|1/1 [00:00, 26.14file/s]

To track the changes with git, run:

	git add artifacts/model.joblib.dvc

To enable auto staging, run:

	dvc config core.autostage true
[master 26ec045] Trained model 1
 2 files changed, 6 insertions(+)
 create mode 100644 artifacts/.gitignore
 create mode 100644 artifacts/model.joblib.dvc


In [48]:
!git add Homework_Pipeline.ipynb iris.csv.dvc

In [ ]:
!git commit -m "Trained model 1"
!git tag -a "v1.0"

## Add rows

In [58]:
df=load_iris(as_frame=True) #from load_iris
new_rows=df.frame.sample(10,replace=True)
new_rows['target'] = new_rows['target'].map({i: name for i, name in enumerate(df.target_names)})
new_rows=new_rows.rename(columns={'sepal length (cm)':'sepal_length',
                                  'sepal width (cm)':'sepal_width',
                                 'petal length (cm)':'petal_length',
                                 'petal width (cm)':'petal_width',
                                 'target':'species'})
iris_df_v2=pd.concat([iris_df_v1,new_rows])
iris_df_v2.tail()

,sepal_length,sepal_width,petal_length,petal_width,species
149,5.9,3.0,5.1,1.8,virginica
98,5.1,2.5,3.0,1.1,versicolor
122,7.7,2.8,6.7,2.0,virginica
118,7.7,2.6,6.9,2.3,virginica
107,7.3,2.9,6.3,1.8,virginica


In [59]:
len(iris_df_v2)

160

In [60]:
iris_df_v2.to_csv("iris.csv",index=False)
!dvc add iris.csv

 ⠋ Checking graph
Adding...                                                                       
!
                                                                                
!
  0% Checking cache in '/home/jupyter/.dvc/cache/files/md5'| |0/? [00:00<?,    ?
                                                                                
!
  0%|          |Adding iris.csv to cache              0/1 [00:00<?,     ?file/s]
                                                                                
!
  0%|          |Checking out /home/jupyter/iris.csv   0/1 [00:00<?,    ?files/s]
100% Adding...|████████████████████████████████████████|1/1 [00:00, 24.49file/s]

To track the changes with git, run:

	git add iris.csv.dvc

To enable auto staging, run:

	dvc config core.autostage true


In [61]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from pandas.plotting import parallel_coordinates
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn import metrics

In [62]:
data=iris_df_v2
train, test = train_test_split(data, test_size = 0.4, stratify = data['species'], random_state = 42)
X_train = train[['sepal_length','sepal_width','petal_length','petal_width']]
y_train = train.species
X_test = test[['sepal_length','sepal_width','petal_length','petal_width']]
y_test = test.species

In [63]:
mod_dt = DecisionTreeClassifier(max_depth = 3, random_state = 1)
mod_dt.fit(X_train,y_train)
prediction=mod_dt.predict(X_test)
print('The accuracy of the Decision Tree is',"{:.3f}".format(metrics.accuracy_score(prediction,y_test)))

The accuracy of the Decision Tree is 0.891


In [64]:
iris_df_v2.to_csv("iris.csv",index=False)

In [ ]:
!dvc add artifacts/model.joblib
!dvc add iris.csv
!git add artifacts/.gitignore artifacts/model.joblib.dvc Homework_Pipeline.ipynb iris.csv.dvc
!git commit -m "Trained model 2"
!git tag -a "v2.0"